In [1]:
# ================================================================
#  LCCN-R24b - TRANSLITERATION BASELINES: DEEP AND TRANSFORMERS  [GPU, ~7 h]
#  BiLSTM and CNN-BiLSTM (3 seeds) on the three arms; XLM-RoBERTa and
#  MuRIL at seed 42 on indicxlit and itrans (Table 3 convention);
#  XLM-RoBERTa seeds 43 and 44 on indicxlit for the per-seed Table 8
#  row raw vs indicxlit, written as __seed{N}.csv files.
#  Add Input: lccn-r24a-xlit-arms (output: xlit_texts.csv),
#             paper1-nb1-lccn-final (output), paper1-nb2-lccn-ml (output).
#  Accelerator GPU. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys, os, shutil, glob
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q scikit-learn xgboost scipy')
sh('pip install -q transformers accelerate --upgrade')
for p in ("trainer.py", "/kaggle/working/trainer.py"):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("__pycache__", ignore_errors=True); sys.modules.pop("trainer", None)
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
assert tc, "attach the paper1-nb2-lccn-ml output"
shutil.copy(tc[0], "/kaggle/working/trainer.py")
import torch
assert torch.cuda.is_available(), "GPU is off - Settings > Accelerator > GPU"
print("gpu:", torch.cuda.get_device_name(0))


+ pip install -q scikit-learn xgboost scipy
+ pip install -q transformers accelerate --upgrade
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 36.9 MB/s eta 0:00:00
gpu: Tesla T4


In [2]:
# ================================================================
#  CELL 1 - SETUP AND ARM JOIN
# ================================================================
import importlib, trainer
importlib.reload(trainer)
from trainer import *
import pandas as pd, numpy as np
FP = trainer.fingerprint(); assert trainer.TRAINER_VERSION == "1.4" and FP == "5b607248090bd26e", FP
sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
tr, va, te, HAVE = load_splits(sd[0])
xt = glob.glob("/kaggle/input/**/xlit_texts.csv", recursive=True); assert xt, "attach lccn-r24a output"
g = pd.read_csv(xt[0], dtype=str, keep_default_na=False); g.columns = [c.strip().lower() for c in g.columns]
idc = next(c for c in g.columns if c in ("id", "comment_id"))
ARMS_NEW = ["indicxlit", "itrans", "aksharamukha_itrans"]
for f in (tr, va, te):
    ids = f["id"].astype(str)
    for a in ARMS_NEW:
        f[f"text_{a}"] = ids.map(dict(zip(g[idc].astype(str), g[f"text_{a}"]))); assert f[f"text_{a}"].notna().all(), a
print("arms joined:", ARMS_NEW)


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']
arms joined: ['indicxlit', 'itrans', 'aksharamukha_itrans']


In [3]:
# ================================================================
#  CELL 2 - DEEP (3 seeds), TRANSFORMERS SEED 42, XLM-R EXTRA SEEDS
# ================================================================
deep = run_grid(tr, va, te, ARMS_NEW, ["BiLSTM", "CNN-BiLSTM"], tag="lccn_r24_xlit_deep", resume=True)
print(deep.to_string(index=False))
tf42 = run_grid(tr, va, te, ["indicxlit", "itrans"], ["XLM-RoBERTa", "MuRIL"],
                tag="lccn_r24_xlit_tf42", seeds_tf=(42,), resume=True)
print(tf42.to_string(index=False))
for m in ("XLM-RoBERTa", "MuRIL"):
    for a in ("indicxlit", "itrans"):
        s = f"/kaggle/working/preds/{m}__{a}.csv"
        if os.path.exists(s): shutil.copy(s, f"/kaggle/working/preds/{m}__{a}__seed42.csv")

have = {os.path.basename(f) for f in glob.glob("/kaggle/input/**/preds/*__seed*.csv", recursive=True)
        + glob.glob("/kaggle/working/preds/*__seed*.csv")}
rows = []
for seed in (43, 44):
    tag_file = f"XLM-RoBERTa__indicxlit__seed{seed}.csv"
    if tag_file in have: print("skip", tag_file); continue
    p = "/kaggle/working/preds/XLM-RoBERTa__indicxlit.csv"
    if os.path.exists(p): os.remove(p)
    r = run_grid(tr, va, te, ["indicxlit"], ["XLM-RoBERTa"], tag=f"lccn_r24_indicxlit_seed{seed}", seeds_tf=(seed,), resume=False)
    shutil.copy(p, f"/kaggle/working/preds/{tag_file}")
    rows.append(dict(arm="indicxlit", seed=seed, macro_f1=round(float(r.iloc[0]["indicxlit"]), 4)))
pd.DataFrame(rows).to_csv("/kaggle/working/results_lccn_r24_indicxlit_seeds.csv", index=False)
print(pd.DataFrame(rows).to_string(index=False))

print("\nReference seed-42 values from NB4/NB5: raw XLM-R 0.8058, lccn_indicxlit XLM-R 0.8130; raw MuRIL and lccn_indicxlit MuRIL from Table 3.")
print("DONE - results_lccn_r24_xlit_deep.csv, results_lccn_r24_xlit_tf42.csv, preds/ written.")
print("Attach this output to lccn-r13-perseed-stats (with COMPARISONS extended by ('raw','indicxlit')) and re-run.")


resume: 35 prediction files already on disk


=== BiLSTM / indicxlit ===
  -> macro-F1 0.7612 +/- 0.0039

=== BiLSTM / itrans ===
  -> macro-F1 0.7670 +/- 0.0013

=== BiLSTM / aksharamukha_itrans ===
  -> macro-F1 0.7706 +/- 0.0039

=== CNN-BiLSTM / indicxlit ===
  -> macro-F1 0.7503 +/- 0.0083

=== CNN-BiLSTM / itrans ===
  -> macro-F1 0.7580 +/- 0.0036

=== CNN-BiLSTM / aksharamukha_itrans ===
  -> macro-F1 0.7600 +/- 0.0025

wrote /kaggle/working/results_lccn_r24_xlit_deep.csv
     model family  indicxlit  indicxlit_std  itrans  itrans_std  aksharamukha_itrans  aksharamukha_itrans_std
    BiLSTM   deep     0.7612         0.0039   0.767      0.0013               0.7706                   0.0039
CNN-BiLSTM   deep     0.7503         0.0083   0.758      0.0036               0.7600                   0.0025
resume: 41 prediction files already on disk


=== XLM-RoBERTa / indicxlit ===


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7158
    epoch  2/10  val macro-F1 = 0.8059
    epoch  3/10  val macro-F1 = 0.8122
    epoch  4/10  val macro-F1 = 0.8288
    epoch  5/10  val macro-F1 = 0.8245
    epoch  6/10  val macro-F1 = 0.8163
    epoch  7/10  val macro-F1 = 0.8216
    early stop at epoch 7
  -> macro-F1 0.8112

wrote /kaggle/working/results_lccn_r24_indicxlit_seed43.csv

=== XLM-RoBERTa / indicxlit ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7508
    epoch  2/10  val macro-F1 = 0.7997
    epoch  3/10  val macro-F1 = 0.8211
    epoch  4/10  val macro-F1 = 0.8151
    epoch  5/10  val macro-F1 = 0.8285
    epoch  6/10  val macro-F1 = 0.8299
    epoch  7/10  val macro-F1 = 0.8260
    epoch  8/10  val macro-F1 = 0.8249
    epoch  9/10  val macro-F1 = 0.8221
    early stop at epoch 9
  -> macro-F1 0.8209

wrote /kaggle/working/results_lccn_r24_indicxlit_seed44.csv
      arm  seed  macro_f1
indicxlit    43    0.8112
indicxlit    44    0.8209

Reference seed-42 values from NB4/NB5: raw XLM-R 0.8058, lccn_indicxlit XLM-R 0.8130; raw MuRIL and lccn_indicxlit MuRIL from Table 3.
DONE - results_lccn_r24_xlit_deep.csv, results_lccn_r24_xlit_tf42.csv, preds/ written.
Attach this output to lccn-r13-perseed-stats (with COMPARISONS extended by ('raw','indicxlit')) and re-run.
